# Notebook per Pre-training Self-Supervised (SpectraMAE / HybridMAE)

Questo notebook implementa il pre-training di un Masked Autoencoder per spettri Raman.
Supporta sia l'architettura **SpectraMAE** (solo Transformer) sia l'architettura **HybridMAE** (CNN + Transformer).
Il modello e tutti gli iperparametri vengono caricati da un file `.yaml`, garantendo riproducibilità e flessibilità.

**Struttura del Notebook:**
1. **Setup**: Librerie, seed globale, path di progetto.
2. **Configurazione**: Lettura del file `.yaml` (passato via env var `EXP_CONFIG_FILE` o default).
3. **Caricamento Dati**: Dataset di pre-training (solo spettri, senza etichette).
4. **Costruzione Modello**: Factory da config (`Spectra_MAE` o `HybridMAE`).
5. **Training Engine**: Loop MAE unificato che gestisce entrambe le architetture.
6. **Esecuzione**: Avvio training e salvataggio checkpoint.
7. **Visualizzazione**: Curve di loss e spettri originali vs ricostruiti.

In [ ]:
# --- CELL 1: ENVIRONMENT SETUP & IMPORTS ---
import sys
import os
import yaml
import numpy as np
import torch
import matplotlib.pyplot as plt
from torch.utils.data import TensorDataset, DataLoader
from src.data.augmentation import RamanDataset

# ========== GLOBAL SEED FOR REPRODUCIBILITY ==========
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
print(f'✅ Global seed set to {SEED} for reproducibility')
# =====================================================

PROJECT_PATH = os.path.abspath(os.path.join(os.getcwd(), ".."))
if PROJECT_PATH not in sys.path:
    sys.path.insert(0, PROJECT_PATH)
os.chdir(PROJECT_PATH)
print(f'Working dir: {os.getcwd()}')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'✅ Setup completed! Using device: {device}')

In [ ]:
# --- CELL 2: CONFIGURATION LOADING ---
# Prende il config dalla shell, oppure usa il default HybridMAE se lanciato a mano
# Per Spectra_MAE: configs/pretraining/exp_01_smae_baseline.yaml
# Per HybridMAE:   configs/pretraining/exp_02_hybrid_mae_baseline.yaml
import shutil

# Default config if not launched via bash script
config_file = os.environ.get('EXP_CONFIG_FILE', 'configs/pretraining/exp_01_smae_baseline.yaml')
assert os.path.exists(config_file), f'Config not found: {config_file}'

with open(config_file, 'r') as f:
    config = yaml.safe_load(f)

ARCHITECTURE = config.get('model', {}).get('architecture', 'Spectra_MAE')
print(f'🏗️ Selected Architecture: {ARCHITECTURE}')

output_dir = config_file.replace('configs', 'experiments').replace('.yaml', '')
os.makedirs(output_dir, exist_ok=True)
shutil.copy(config_file, os.path.join(output_dir, 'config_usato.yaml'))

print(f"Running Experiment: {config.get('experiment_name', 'unknown')}")
print(f"Results will be saved in:\n{output_dir}")




In [ ]:
# 🚀 SOTA FIX: ONLY Train (100%). No wasted Test or Validation sets!
tensor_x_train = torch.tensor(X_data, dtype=torch.float32).unsqueeze(1)

BATCH_SIZE = int(ds_cfg.get('batch_size', config.get('training', {}).get('batch_size', 256)))

train_loader = DataLoader(RamanDataset(tensor_x_train, is_train=True, noise_std=0.01, scale_range=0.05, baseline_shift=0.05), batch_size=BATCH_SIZE, shuffle=True,  drop_last=True)
val_loader   = None

print("-" * 50)
print(f"🚀 DATA SPLIT COMPLETED (Zero Waste Policy)")
print(f"   Train: {N} spectra (100%)")
print(f"   Val:   None (100% used for training)")
print(f"   Batch Size: {BATCH_SIZE}")
print("-" * 50)


In [ ]:
# --- CELL 4: MODEL CONSTRUCTION ---
from src.models.factory import build_model_from_config

model = build_model_from_config(config, device)
n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"✅ Model '{ARCHITECTURE}' built successfully.")
print(f"   Trainable parameters: {n_params:,}")

In [ ]:
# --- CELL 5: TRAINING ENGINE IMPORT ---
from src.engine.pretrain import train_mae
print(f"✅ Advanced training engine (with Cosine Scheduler and Similarity) imported for '{ARCHITECTURE}'.")

In [ ]:
# --- CELL 6: EXECUTE TRAINING ---
import json

print("="*50)
print(f"=== PRE-TRAINING PHASE: {ARCHITECTURE} ===")
print("="*50)

best_weights, history = train_mae(model, train_loader, val_loader, config, device, ARCHITECTURE)

model_save_path = os.path.join(output_dir, "best_pretrained_model.pth")
torch.save(best_weights, model_save_path)
print(f"✅ Pre-trained model saved to: {model_save_path}")

history_path = os.path.join(output_dir, "pretrain_history.json")
with open(history_path, 'w') as f:
    json.dump(history, f, indent=2)
print(f"✅ History saved to: {history_path}")

In [ ]:
# --- CELL 7: RESULTS VISUALIZATION ---
from src.utils.plotting import plot_mae_history_sota, plot_mae_reconstruction_sota

report_dir = os.path.join(output_dir, "report")

# ---- 7a: Loss and Cosine Similarity Curves ----
plot_mae_history_sota(history, report_dir)

# ---- 7b: Reconstruction Examples ----
print("\n--- Visualization of SOTA Reconstruction Examples ---")
model.load_state_dict(best_weights)
plot_mae_reconstruction_sota(model, val_loader, device, ARCHITECTURE, report_dir, n_examples=4)

print("✅ Training and qualitative analysis completed successfully.")